In [2]:
import pandas as pd
import spacy
from collections import Counter

corpus = pd.read_csv("resenas_clean.csv")
nlp = spacy.load("es_core_news_md")

corpus.head()

,texto,calificacion,polaridad,tipo_lugar,fuente,fecha,texto_limpio
0,"Un lugar espectacular y, sin duda, una visita ...",3,neutral,parque,google_maps,06/06/2026 02:10:30,un lugar espectacular y sin duda una visita ob...
1,"El parque es espectacular, las 3 son porque es...",3,neutral,parque,google_maps,06/05/2026 21:24:50,el parque es espectacular las son porque esta ...
2,"Increíble!! Lleno de flora y fauna del país, c...",5,positivo,parque,google_maps,06/05/2026 03:38:27,increíble lleno de flora y fauna del país con ...
3,Parque imperdível! O estacionamento fora do pa...,5,positivo,parque,google_maps,06/02/2026 17:46:50,parque imperdível o estacionamento fora do par...
4,Fabuloso lugar para apreciar y tener un contac...,5,positivo,parque,google_maps,06/02/2026 16:26:50,fabuloso lugar para apreciar y tener un contac...


In [19]:
#cuenta la cantidad de palabras 
corpus["cantidad_palabras"] = corpus["texto_limpio"].apply(lambda x: len(x.split()))
#cuenta la cantidad de caracteres
corpus["cantidad_caracteres"] = corpus["texto_limpio"].apply(len)

corpus[["cantidad_palabras", "cantidad_caracteres"]].describe().round(2)

,cantidad_palabras,cantidad_caracteres
count,1008.00,1008.00
mean,206.50,1194.52
std,160.12,916.70
min,2.00,13.00
25%,110.00,659.25
50%,158.00,935.50
75%,273.25,1561.00
max,883.00,5014.00


In [ ]:
# Une todos los textos del corpus 
palabras = " ".join(corpus["texto_limpio"]).split()

# Counter cuenta cuántas veces aparece cada palabra
frecuencia = Counter(palabras)

#  devuelve las 15 palabras más repetidas
palabras_frecuentes = pd.DataFrame(
    frecuencia.most_common(15),
    columns=["palabra", "frecuencia"]
)

palabras_frecuentes

,palabra,frecuencia
0,de,12558
1,la,7547
2,que,7132
3,y,5660
4,en,5190
5,el,4359
6,a,3658
7,un,3445
8,es,3313
9,una,3184


In [7]:
# Aplica el pipeline completo de spaCy a cada reseña
corpus["spacy_doc"] = corpus["texto_limpio"].apply(nlp)

In [ ]:
# Cuenta las etiquetas POS de todo el corpus
pos_counter = Counter(
    token.pos_
    for doc in corpus["spacy_doc"]
    for token in doc
    if not token.is_space
)

total_tokens = sum(pos_counter.values())

# Convierte el contador a DataFrame 
distribucion_pos = pd.DataFrame(
    list(pos_counter.items()),        
    columns=["categoría_pos", "cantidad"]
)

# Ordena de mayor a menor por cantidad
distribucion_pos = distribucion_pos.sort_values("cantidad", ascending=False)

# Agrega el porcentaje
distribucion_pos["porcentaje"] = round(
    (distribucion_pos["cantidad"] / total_tokens) * 100, 2
)

distribucion_pos

,categoría_pos,cantidad,porcentaje
1,NOUN,43347,20.82
0,DET,31022,14.90
4,ADP,30320,14.57
6,VERB,22051,10.59
2,ADJ,19291,9.27
5,PRON,14175,6.81
9,ADV,13364,6.42
8,AUX,11146,5.35
3,CCONJ,8037,3.86
7,PROPN,7798,3.75


In [ ]:
#cuenta los valores de polaridad
corpus["polaridad"].value_counts()

polaridad
positivo    542
negativo    455
neutral      11
Name: count, dtype: int64

In [14]:
def contar_pos(documentos):

    contador = Counter()

    for doc in documentos:
        for token in doc:
            if not token.is_space:
                contador[token.pos_] += 1

    return contador

In [ ]:
#filtra por reseñas positivas y negativas 
positivas = corpus[corpus["polaridad"] == "positivo"]

negativas = corpus[corpus["polaridad"] == "negativo"]

In [ ]:
#llama a la funcion pasandole los documentos de las reseñas positivas
pos_positivas = contar_pos(positivas["spacy_doc"])

pos_negativas = contar_pos(negativas["spacy_doc"])

In [ ]:
categorias = ["NOUN", "VERB", "ADJ", "ADV", "PRON"]

comparacion_pos = pd.DataFrame({
    "POS":       categorias,
    "Positivas": [pos_positivas[c] for c in categorias],
    "Negativas": [pos_negativas[c] for c in categorias]
})

comparacion_pos

,POS,Positivas,Negativas
0,NOUN,22358,20761
1,VERB,10891,11034
2,ADJ,10136,9067
3,ADV,6343,6921
4,PRON,7107,6986
